# Dataset Loaders
This notebook builds the full 500‑email dataset from the sources listed in the PDF.
We keep the same target schema as the demo notebook.

In [19]:
from __future__ import annotations

import re
from typing import List, Dict
from pathlib import Path

import pandas as pd

SCHEMA_COLUMNS = [
    "email_id" ,
    "source" ,
    "actual_class" ,
    "sender_address" ,
    "subject_line" ,
    "body_content" ,
    "extracted_links" ,
]

URL_REGEX = re.compile(r"https?://[^\s<>]+", re.IGNORECASE)

def extract_links(text: str) -> List[str]:
    if not text:
        return []
    return URL_REGEX.findall(text)

def normalize_record(
    email_id: str,
    source: str,
    actual_class: int,
    sender_address: str,
    subject_line: str,
    body_content: str,
) -> Dict[str, object]:
    body_content = body_content or ""
    subject_line = subject_line or ""
    extracted_links = extract_links(body_content)
    return {
        "email_id": email_id,
        "source": source,
        "actual_class": int(actual_class),
        "sender_address": sender_address or "" ,
        "subject_line": subject_line,
        "body_content": body_content,
        "extracted_links": extracted_links,
    }

def validate_schema(df: pd.DataFrame) -> None:
    missing = [c for c in SCHEMA_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

## Local Data Configuration
All datasets are loaded from local CSV files (no Kaggle/URL downloads).

In [20]:
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Local data file paths (all stored in data/ directory)
# SpamAssassin benign emails (raw email files, not CSV)
SPAMASSASSIN_DIR = Path("..") / "data" / "raw" / "spamassassin" / "easy_ham"

# Phishing sources
PHISHBOWL_PATH = DATA_DIR / "cornell_phishbowl_data.csv"
LEGACY_LLM_PHISHING_PATH = DATA_DIR / "legacy_llm_phishing.csv"
HYBRID_PATH = DATA_DIR / "vtriad_hybrid_diverse.csv"

# Verify all required paths exist
REQUIRED_PATHS = {
    "SpamAssassin (benign)": SPAMASSASSIN_DIR,
    "Phishbowl": PHISHBOWL_PATH,
    "Legacy LLM Phishing": LEGACY_LLM_PHISHING_PATH,
    "Hybrid": HYBRID_PATH,
}

print("=== LOCAL DATA PATHS ===")
missing_paths = []
for name, path in REQUIRED_PATHS.items():
    exists = path.exists()
    status = "✓" if exists else "✗"
    path_type = "(dir)" if path.is_dir() else "(file)" if exists else ""
    print(f"{status} {name}: {path.name} {path_type}")
    if not exists:
        missing_paths.append(f"{name} ({path})")

if missing_paths:
    print(f"\n⚠ Missing paths:\n  - " + "\n  - ".join(missing_paths))
else:
    print("\n✓ All required paths found locally!")

=== LOCAL DATA PATHS ===
✓ SpamAssassin (benign): easy_ham (dir)
✓ Phishbowl: cornell_phishbowl_data.csv (file)
✓ Legacy LLM Phishing: legacy_llm_phishing.csv (file)
✓ Hybrid: vtriad_hybrid_diverse.csv (file)

✓ All required paths found locally!


In [21]:
def load_spamassassin(email_dir: Path, limit: int = 100) -> pd.DataFrame:
    """
    Load SpamAssassin benign emails from raw .eml files.
    Searches recursively for email files (ignores __MACOSX and hidden dirs).
    
    Args:
        email_dir: Directory containing email files (e.g., easy_ham/)
        limit: Max number of emails to load
    """
    import email
    from email import policy
    
    rows = []
    email_files = []
    
    # Find all files recursively, skip __MACOSX and hidden directories
    for item in email_dir.rglob("*"):
        if '__MACOSX' in str(item) or '.DS_Store' in item.name:
            continue
        if item.is_file():
            email_files.append(item)
            if len(email_files) >= limit:
                break
    
    print(f"    Found {len(email_files)} email files, parsing...")
    parsed_count = 0
    
    for idx, email_file in enumerate(email_files):
        try:
            with open(email_file, 'r', encoding='utf-8', errors='ignore') as f:
                msg = email.message_from_file(f, policy=policy.default)
                
                # Extract email parts
                sender = msg.get('From', '')
                subject = msg.get('Subject', '')
                
                # Get body (handle multipart)
                body = ''
                if msg.is_multipart():
                    for part in msg.iter_parts():
                        if part.get_content_type() == 'text/plain':
                            body = part.get_payload(decode=True)
                            if isinstance(body, bytes):
                                body = body.decode('utf-8', errors='ignore')
                            break
                else:
                    body = msg.get_payload(decode=True)
                    if isinstance(body, bytes):
                        body = body.decode('utf-8', errors='ignore')
                
                rows.append(normalize_record(
                    email_id=f"SA_{parsed_count+1}",
                    source='SpamAssassin',
                    actual_class=0,  # benign
                    sender_address=sender,
                    subject_line=subject,
                    body_content=body,
                ))
                parsed_count += 1
                
        except Exception as e:
            pass  # Silently skip unparseable files
    
    print(f"    Successfully parsed {parsed_count} emails")
    return pd.DataFrame(rows)


def load_phishbowl(path: Path, limit: int = 100) -> pd.DataFrame:
    """Load Phishbowl phishing dataset (CSV: title, email_message)."""
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        if len(rows) >= limit:
            break
        rows.append(normalize_record(
            email_id=f"PB_{i+1}",
            source='Phishbowl',
            actual_class=1,  # phishing
            sender_address=row.get('Unnamed: 0', '') if 'Unnamed: 0' in df.columns else '',
            subject_line=row.get('title', ''),
            body_content=row.get('email_message', ''),
        ))
    return pd.DataFrame(rows)


def load_legacy_llm_phishing(path: Path) -> pd.DataFrame:
    """Load legacy LLM phishing dataset (CSV: subject, sender, body)."""
    if not path.exists():
        return pd.DataFrame(columns=SCHEMA_COLUMNS)
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"LLM_{i+1}",
            source='Legacy_LLM_Phishing',
            actual_class=1,  # phishing
            sender_address=row.get('sender', ''),
            subject_line=row.get('subject', ''),
            body_content=row.get('body', ''),
        ))
    return pd.DataFrame(rows)


def load_hybrid(path: Path) -> pd.DataFrame:
    """Load hybrid phishing dataset (CSV: subject, sender, body)."""
    if not path.exists():
        return pd.DataFrame(columns=SCHEMA_COLUMNS)
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"HYB_{i+1}",
            source='Self_Generated_Hybrid',
            actual_class=1,  # phishing
            sender_address=row.get('sender', ''),
            subject_line=row.get('subject', ''),
            body_content=row.get('body', ''),
        ))
    return pd.DataFrame(rows)


print("✓ All loader functions defined")

✓ All loader functions defined


In [22]:
def build_master_dataset() -> pd.DataFrame:
    """
    Build master dataset from LOCAL sources only.
    No Kaggle API, no URL downloads - all data is kept local.
    """
    
    print("=== BUILDING MASTER DATASET (LOCAL FILES) ===\n")
    
    frames = []
    
    # Load SpamAssassin (benign) - raw email files
    if SPAMASSASSIN_DIR.exists():
        print(f"Loading SpamAssassin from: {SPAMASSASSIN_DIR.name}")
        sa_df = load_spamassassin(SPAMASSASSIN_DIR, limit=100)
        print(f"  ✓ Loaded {len(sa_df)} emails, body avg: {sa_df['body_content'].str.len().mean():.0f} chars\n")
        frames.append(sa_df)
    else:
        print(f"  ✗ SpamAssassin not found: {SPAMASSASSIN_DIR}\n")
    
    # Load Phishbowl (real phishing)
    if PHISHBOWL_PATH.exists():
        print(f"Loading Phishbowl from: {PHISHBOWL_PATH.name}")
        pb_df = load_phishbowl(PHISHBOWL_PATH, limit=100)
        print(f"  ✓ Loaded {len(pb_df)} emails, body avg: {pb_df['body_content'].str.len().mean():.0f} chars\n")
        frames.append(pb_df)
    else:
        print(f"  ✗ Phishbowl not found: {PHISHBOWL_PATH}\n")
    
    # Load Legacy LLM Phishing
    if LEGACY_LLM_PHISHING_PATH.exists():
        print(f"Loading Legacy LLM Phishing from: {LEGACY_LLM_PHISHING_PATH.name}")
        llm_df = load_legacy_llm_phishing(LEGACY_LLM_PHISHING_PATH)
        print(f"  ✓ Loaded {len(llm_df)} emails, body avg: {llm_df['body_content'].str.len().mean():.0f} chars\n")
        frames.append(llm_df)
    else:
        print(f"  ✗ Legacy LLM not found: {LEGACY_LLM_PHISHING_PATH}\n")
    
    # Load Hybrid (synthetic)
    if HYBRID_PATH.exists():
        print(f"Loading Hybrid from: {HYBRID_PATH.name}")
        hyb_df = load_hybrid(HYBRID_PATH)
        print(f"  ✓ Loaded {len(hyb_df)} emails, body avg: {hyb_df['body_content'].str.len().mean():.0f} chars\n")
        frames.append(hyb_df)
    else:
        print(f"  ✗ Hybrid not found: {HYBRID_PATH}\n")
    
    # Combine all frames
    if not frames:
        raise ValueError("No datasets loaded! Check file paths.")
    
    master = pd.concat(frames, ignore_index=True)
    validate_schema(master)
    
    print(f"=== MASTER DATASET BUILT ===")
    print(f"Total rows: {len(master)}")
    print(f"\nBreakdown by source:")
    print(master['source'].value_counts())
    print(f"\nBreakdown by class:")
    print(master['actual_class'].value_counts())
    print(f"\nAverage body content length by source:")
    print(master.groupby('source')['body_content'].apply(lambda x: x.str.len().mean()).round(0))
    
    return master


# Build the master dataset
print("Loading datasets from local sources...\n")
master_df = build_master_dataset()

# Save for downstream notebooks
output_path = DATA_DIR / "master_legacy.csv"
master_df.to_csv(output_path, index=False)
print(f"\n✓ Saved to: {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")

Loading datasets from local sources...

=== BUILDING MASTER DATASET (LOCAL FILES) ===

Loading SpamAssassin from: easy_ham
    Found 100 email files, parsing...
    Successfully parsed 100 emails
  ✓ Loaded 100 emails, body avg: 1489 chars

Loading Phishbowl from: cornell_phishbowl_data.csv
  ✓ Loaded 100 emails, body avg: 1937 chars

Loading Legacy LLM Phishing from: legacy_llm_phishing.csv
  ✓ Loaded 100 emails, body avg: 183 chars

Loading Hybrid from: vtriad_hybrid_diverse.csv
  ✓ Loaded 48 emails, body avg: 198 chars

=== MASTER DATASET BUILT ===
Total rows: 348

Breakdown by source:
source
SpamAssassin             100
Phishbowl                100
Legacy_LLM_Phishing      100
Self_Generated_Hybrid     48
Name: count, dtype: int64

Breakdown by class:
actual_class
1    248
0    100
Name: count, dtype: int64

Average body content length by source:
source
Legacy_LLM_Phishing       183.0
Phishbowl                1937.0
Self_Generated_Hybrid     198.0
SpamAssassin             1489.0
Na

In [23]:
print(f"Total rows: {len(master_df)}")
print(f"\nBreakdown by source:")
print(master_df['source'].value_counts())
print(f"\nBreakdown by class:")
print(master_df['actual_class'].value_counts())
print(f"\nSample with content:")
print(master_df[master_df['body_content'].str.len() > 0].head(2))

Total rows: 348

Breakdown by source:
source
SpamAssassin             100
Phishbowl                100
Legacy_LLM_Phishing      100
Self_Generated_Hybrid     48
Name: count, dtype: int64

Breakdown by class:
actual_class
1    248
0    100
Name: count, dtype: int64

Sample with content:
  email_id        source  actual_class  \
0     SA_1  SpamAssassin             0   
1     SA_2  SpamAssassin             0   

                              sender_address               subject_line  \
0             Robert Elz <kre@munnari.OZ.AU>   Re: New Sequences Window   
1  Steve Burt <Steve_Burt@cursor-system.com>  [zzzzteana] RE: Alexander   

                                        body_content  \
0      Date:        Wed, 21 Aug 2002 10:54:46 -05...   
1  Martin A posted:\nTassos Papadopoulos, the Gre...   

                                     extracted_links  
0  [https://listman.redhat.com/mailman/listinfo/e...  
1  [http://us.click.yahoo.com/pt6YBB/NXiEAA/mG3HA...  
